In [ ]:
!pip install sentence-transformers faiss-cpu transformers accelerate bitsandbytes

In [96]:
!pip install nltk bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.1 MB/s eta 0:00:00


In [63]:
import os

corpus = []
folder_path = "/content/Research-Chatbot"  # Adjust this to your folder

for file_name in os.listdir(folder_path):
    file_path = os.path.join(folder_path, file_name)
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
            corpus.extend([para.strip() for para in content.split('\n') if len(para.strip()) > 50])
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

print(f"Loaded {len(corpus)} paragraphs.")


Loaded 67 paragraphs.


In [85]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')  # Fast and good for semantic search
corpus_embeddings = model.encode(corpus, show_progress_bar=True, convert_to_numpy=True)


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

In [86]:
import faiss
import numpy as np

embedding_dim = corpus_embeddings.shape[1]  # Should be 384 for all-MiniLM-L6-v2
index = faiss.IndexFlatL2(embedding_dim)
index.add(corpus_embeddings)

print(f"FAISS index built with {index.ntotal} vectors.")


FAISS index built with 67 vectors.


In [87]:
def retrieve_passages(query, top_k=3):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)
    results = [corpus[idx] for idx in indices[0]]
    return results


In [88]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

t5_model = T5ForConditionalGeneration.from_pretrained("t5-base")
t5_tokenizer = T5Tokenizer.from_pretrained("t5-base")

def generate_answer(query, context_passages):
    context = " ".join(context_passages)
    prompt = f"question: {query} context: {context}"
    inputs = t5_tokenizer(prompt, return_tensors="pt", truncation=True, padding=True)
    outputs = t5_model.generate(**inputs, max_length=128)
    answer = t5_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer


In [89]:
query = input("What are library opening hours?")
passages = retrieve_passages(query)

print("\nTop Retrieved Passages:")
for i, p in enumerate(passages):
    print(f"[{i+1}] {p}\n")

answer = generate_answer(query, passages)
print("Generated Answer:\n", answer)


What are library opening hours?


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



Top Retrieved Passages:
[1] "DBS students raise €4,030 for Temple St. Children's Hospital","DBS students held a themed charity event in aid of Temple St. Children's Hospital. The event was spearheaded by\n\nProject Management students with the intention of bridging communication between college students, alumni and industry. The purpose of the event was to give students the opportunity to network amongst leading global entrepreneurs and break down barriers between boardrooms and classrooms. The project team provided a platform for successful entrepreneurs to voice their journey to young aspiring individuals.",

[2] "Student Entertainment",    "The Student Experience Team, in conjunction with our Student Union, organise a full and varied schedule of social and cultural events throughout the year. From Freshers week in September, RAG week, weekly film screenings, cultural excursions and day trips, and the Formal Ball and Awards in May, there is something for everyone.",

[3] "DBS studen

In [90]:
def start_chatbot(top_k=3):
    print("🎓 College Brochure Chatbot is ready! Type 'exit' to quit.\n")

    while True:
        query = input("You: ")
        if query.lower() in ["exit", "quit"]:
            print("Goodbye!")
            break

        retrieved_passages = retrieve_passages(query)
        answer = generate_answer(query,retrieved_passages)

        # Step 5: Show the answer
        print("🤖 Chatbot:", answer)
        print("-" * 60)

In [69]:
start_chatbot()


🎓 College Brochure Chatbot is ready! Type 'exit' to quit.

You: hi


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


🤖 Chatbot: Kieran O'Shea, from Decathlon spoke to students about the opening of the sports retail store
------------------------------------------------------------
You: How many books DBS Library has?
🤖 Chatbot: over 43,000
------------------------------------------------------------
You: quit
Goodbye!


In [91]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Load the TinyLLaMA Chat model
model_id_tinyllm = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer_tinyllm = AutoTokenizer.from_pretrained(model_id_tinyllm)
model_tinyllm = AutoModelForCausalLM.from_pretrained(model_id_tinyllm, device_map="auto", torch_dtype="auto")

In [92]:
def refine_answer_llama(query, raw_answer):
    prompt = f"""<|system|>You are a helpful assistant that turns answers into full, natural sentences.<|end|>
<|user|>Question: {query}
Answer: {raw_answer}
Please rephrase this as a full sentence.<|end|>
<|assistant|>"""

    inputs = tokenizer_tinyllm(prompt, return_tensors="pt").to(model_tinyllm.device)
    outputs = model_tinyllm.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.8,
        pad_token_id=tokenizer_tinyllm.eos_token_id
    )

    full_output = tokenizer_tinyllm.decode(outputs[0], skip_special_tokens=True)
    # Extract the assistant's response
    if "<|assistant|>" in full_output:
        return full_output.split("<|assistant|>")[-1].strip()
    return full_output.strip()


In [93]:
query = "How many books does the DBS Library have?"
raw_answer = "over 43,000"

refined = refine_answer_llama(query, raw_answer)
print("📘 Refined Answer:", refined)

📘 Refined Answer: The DBS Library has over 43,000 books.


In [94]:
def start_chatbot_v2(top_k=3):
    print("🎓 College Brochure Chatbot is ready! Type 'exit' to quit.\n")

    while True:
        query = input("You: ")
        if query.lower() in ["exit", "quit"]:
            print("Goodbye!")
            break

        retrieved_passages = retrieve_passages(query)
        answer = generate_answer(query,retrieved_passages)
        print("🤖 Chatbot original answer:", answer)
        print("-" * 60)

        refined_answer = refine_answer_llama(query, answer)

        # Step 5: Show the answer
        print("🤖 Chatbot refined answer:", refined_answer)
        print("-" * 60)

In [95]:
start_chatbot_v2()

🎓 College Brochure Chatbot is ready! Type 'exit' to quit.

You: hi
🤖 Chatbot original answer: Kieran O'Shea, from Decathlon spoke to students about the opening of the sports retail store
------------------------------------------------------------
🤖 Chatbot refined answer: Kieran O'Shea, from Decathlon, spoke to students about the opening of the sports retail store.
------------------------------------------------------------
You: How many books DBS Library has?
🤖 Chatbot original answer: over 43,000
------------------------------------------------------------
🤖 Chatbot refined answer: There are 43,000 books in the DBS Library.
------------------------------------------------------------
You: quit
Goodbye!


Evaluation Pipeline

In [98]:
import nltk
nltk.download('punkt')

import time
from sklearn.metrics import accuracy_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bert_score

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [103]:
test_set = [
    {"query": "How many books DBS Library has?", "expected_answer": "over 43,000"},
     {"query": "Where can I access DBS WiFi?", "expected_answer": ""},
      {"query": "What are library opening hours?", "expected_answer": " 24 hours a day"},
       {"query": "What ratings did DBS earned?", "expected_answer": " 4 stars"},
        {"query": "how to view Library account?", "expected_answer": ""},
         {"query": "Are the Guides to Library resources for students with disabilities are also available in the Library?", "expected_answer": " on the library website"},
          {"query": "How many university partnerships does DBS has developed?", "expected_answer": " over 75"}
  ]

In [99]:


smoothie = SmoothingFunction().method4

def evaluate_rag_model(test_set):
    results = []
    total_time = 0
    all_generated = []
    all_expected = []

    for i, item in enumerate(test_set):
        query = item["query"]
        expected = item["expected_answer"]

        start_time = time.time()
        retrieved_passages = retrieve_passages(query)
        generated = generate_answer(query,retrieved_passages)
        end_time = time.time()

        response_time = end_time - start_time
        total_time += response_time

        # Save for BERTScore later
        all_generated.append(generated)
        all_expected.append(expected)

        # Exact match
        exact_match = int(expected.lower() in generated.lower())

        # BLEU Score
        reference = [expected.split()]
        candidate = generated.split()
        bleu = sentence_bleu(reference, candidate, smoothing_function=smoothie)

        results.append({
            "Query": query,
            "Generated": generated,
            "Expected": expected,
            "ExactMatch": exact_match,
            "BLEU": bleu,
            "TimeTaken": response_time
        })

    # BERTScore
    P, R, F1 = bert_score(all_generated, all_expected, lang="en", verbose=True)
    avg_bertscore_f1 = F1.mean().item()

    # Summary metrics
    accuracy = sum(r["ExactMatch"] for r in results) / len(results)
    avg_bleu = sum(r["BLEU"] for r in results) / len(results)
    avg_time = total_time / len(results)

    print(f"\n--- Evaluation Summary ---")
    print(f"Accuracy (Exact Match): {accuracy:.2f}")
    print(f"Average BLEU Score: {avg_bleu:.2f}")
    print(f"Average BERTScore F1: {avg_bertscore_f1:.2f}")
    print(f"Average Inference Time: {avg_time:.2f} seconds\n")

    return results


In [104]:
results = evaluate_rag_model(test_set)
results

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 2.75 seconds, 2.54 sentences/sec

--- Evaluation Summary ---
Accuracy (Exact Match): 0.43
Average BLEU Score: 0.06
Average BERTScore F1: 0.65
Average Inference Time: 3.44 seconds



[{'Query': 'How many books DBS Library has?',
  'Generated': 'over 43,000',
  'Expected': 'over 43,000',
  'ExactMatch': 1,
  'BLEU': 0.2213885886251307,
  'TimeTaken': 2.968395709991455},
 {'Query': 'Where can I access DBS WiFi?',
  'Generated': 'https://books.dbs.ie',
  'Expected': '',
  'ExactMatch': 1,
  'BLEU': 0,
  'TimeTaken': 2.4440271854400635},
 {'Query': 'What are library opening hours?',
  'Generated': 'Monday-Thursday: 09:00-22:00',
  'Expected': ' 24 hours a day',
  'ExactMatch': 0,
  'BLEU': 0,
  'TimeTaken': 1.7966058254241943},
 {'Query': 'What ratings did DBS earned?',
  'Generated': '5 Stars',
  'Expected': ' 4 stars',
  'ExactMatch': 0,
  'BLEU': 0,
  'TimeTaken': 3.813079595565796},
 {'Query': 'how to view Library account?',
  'Generated': 'login to the catalogue',
  'Expected': '',
  'ExactMatch': 1,
  'BLEU': 0,
  'TimeTaken': 2.4499008655548096},
 {'Query': 'Are the Guides to Library resources for students with disabilities are also available in the Library?',
 